<img src="https://iteso.mx/documents/27014/202031/Logo-ITESO-MinimoH.png"
     width="300"/>

# Modelos Garch para predicciones del Valor en riesgo y potencial alcista
### Clase de Modelos no Lineales
##### Por: 
 - **Cesar Santos**
 - **Ivan Morales**
 - **Rafael Takata**

---
### **BP PLC (Anteriormente: British Petroleum)**
<img src="https://cdn.milenio.com/uploads/media/2017/03/09/bp.jpeg"
     align = "center"
     width="300"/>


BP es una de las **"supermajors"** del petróleo y gas a nivel mundial. Fundada en 1909, ha evolucionado de ser una compañía extractora tradicional a buscar una posición de liderazgo en la transición energética. Con sede en Londres, es una empresa centrada en petróleo, gas natural, renovables y movilidad eléctrica.

##### **Modelo de Negocio**

BP divide sus operaciones principalmente en tres segmentos:

1. ``Gasolina y Energías renovables:`` Enfocado en gas natural y proyectos de energía renovable (eólica marina, solar e hidrógeno).

2. ``Producción de petroleo y operaciones:`` La exploración y extracción tradicional de hidrocarburos.

3. ``Clientes y productos:`` Refinación, marketing y su red de estaciones de servicio (retail).

#### Librerías:

In [8]:
import yfinance as yf
import numpy as np
import datetime
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from statsmodels.tsa.stattools import acf, pacf
from arch import arch_model

#### Importamos los datos:

In [2]:
ticker = 'BP'
bp = yf.Ticker(ticker)
df = bp.history(start='2010-01-01', end=datetime.datetime.now().strftime('%Y-%m-%d'))
returns = 100 * np.log(df['Close'] / df['Close'].shift(1)).dropna()

#Visualizamos los retornos
fig = go.Figure()
fig.add_trace(go.Scatter(x=returns.index, y=returns.squeeze(), mode='lines', name='Retornos XOM'))
fig.update_layout(title=f'Retornos Diarios de {ticker}',
                  yaxis_title='Retornos (%)')
fig.show()

Elevamos al cuadrado los retornos para realizar las pruebas estadísticas (ACF y PACF) para ver la viabilidad de utilizar un modelo `GARCH`:

In [3]:
sq_returns = returns ** 2
sig_level = 1.96 / np.sqrt(len(returns))

#acf y pacf
lag_acf = acf(sq_returns, nlags=20)
lag_pacf = pacf(sq_returns, nlags=20, method='ols')

#Visualizamos
fig = make_subplots(rows=1, cols=2, subplot_titles=('ACF de Retornos Cuadrados', 'PACF de Retornos Cuadrados'))
# ACF
fig.add_trace(go.Bar(x=np.arange(len(lag_acf)), y=lag_acf, name='ACF'), row=1, col=1)
# PACF
fig.add_trace(go.Bar(x=np.arange(len(lag_pacf)), y=lag_pacf, name='PACF'), row=1, col=2)

#Significancia
for i in [1, 2]:
    fig.add_hline(y=sig_level, line_dash="dash", line_color="red", row=1, col=i)
    fig.add_hline(y=-sig_level, line_dash="dash", line_color="red", row=1, col=i)

fig.show()

Las gráficas de ACF y PACF muestran barras significativas por lo que se puede asumir que un modelo GARCH es viable.

#### Desarrollamos el modelo:

In [4]:
#Declaramos el modelo GARCH(1,1) con distribución t
am = arch_model(returns, vol='Garch', p=1, q=1, dist='t')
res = am.fit(disp='off')
cond_vol = res.conditional_volatility

#Value at risk al 5% y upside VAR al 95%
q_5 = res.std_resid.quantile(0.05)
q_95 = res.std_resid.quantile(0.95)
mean = res.params['mu']
VaR_5 = mean + cond_vol * q_5
VaR_95 = mean + cond_vol * q_95

# Graficar los Retornos vs el VaR predictivo
fig = go.Figure()
fig.add_trace(go.Scatter(x=returns.index, y=returns.squeeze(), mode='lines',
                         name='Retornos Reales', opacity=0.6))
fig.add_trace(go.Scatter(x=VaR_5.index, y=VaR_5, mode='lines',
                         name='VaR 5% (GARCH)', line=dict(color='red')))
fig.add_trace(go.Scatter(x=VaR_95.index, y=VaR_95, mode='lines',
                         name='VaR 95% (GARCH)', line=dict(color='blue')))

fig.update_layout(title='Retornos de BP vs GARCH(1,1) VaR al 5% y 95%',
                  yaxis_title='Retornos (%)')
fig.show()

#### Veamos el summary para interpretar el modelo:

In [5]:
print(res.summary())

                        Constant Mean - GARCH Model Results                         
Dep. Variable:                        Close   R-squared:                       0.000
Mean Model:                   Constant Mean   Adj. R-squared:                  0.000
Vol Model:                            GARCH   Log-Likelihood:               -7502.91
Distribution:      Standardized Student's t   AIC:                           15015.8
Method:                  Maximum Likelihood   BIC:                           15047.4
                                              No. Observations:                 4072
Date:                      Sun, Mar 15 2026   Df Residuals:                     4071
Time:                              20:40:59   Df Model:                            1
                                 Mean Model                                 
                 coef    std err          t      P>|t|      95.0% Conf. Int.
----------------------------------------------------------------------------
mu  

1. `Omega` ($\omega = 0.0250$): Es la constante del modelo. Representa el nivel base de la varianza si no hubiera choques ni persistencia. Es estadísticamente significativo ($p < 0.05$).

2. `Alpha` ($\alpha = 0.0582$): Es la reacción inmediata al mercado. Indica que el $5.82\%$ de la sorpresa de rentabilidad de ayer (el "choque") se traslada a la volatilidad de hoy. Es un valor bajo, lo que significa que la volatilidad no "salta" de forma violenta ante un solo evento.

3. `Beta` ($\beta = 0.9368$): Es la persistencia. Indica que el $93.68\%$ de la volatilidad de hoy se explica por la volatilidad de ayer. Al ser un valor tan cercano a 1, la volatilidad es muy "memorable": si el mercado está nervioso hoy, lo más probable es que siga nervioso mañana.

Este es un modelo muy común dentro de los mercados financieros, ahora veamos el calculo del Half-life o la vida media de un choque.

Esto se hace a través de la siguiente formula:

$$HL = \frac{\ln(0.5)}{\ln(\alpha + \beta)}$$

En este caso:

$$HL = \frac{\ln(0.5)}{\ln(0.0582 + 0.9368)}$$
 
Despejando:

$$HL = \frac{\ln(0.5)}{\ln(0.995)} = 138.28$$

Por lo tanto, un evento inesperado en el mercado (un "choque") tardará aproximadamente 138 días hábiles en reducir su impacto en la volatilidad a la mitad.

#### Predicción de los siguientes 7 días:

In [15]:
pred_dates = pd.date_range(start=returns.index[-1] + pd.Timedelta(days=1), periods=7)


fig = go.Figure()
# Retornos reales
fig.add_trace(go.Scatter(x=returns[-100:].index, y=returns[-100:].squeeze(), mode='lines',
                         name='Retornos Reales', opacity=0.6))
# Pronóstico VaR 5%
fig.add_trace(go.Scatter(x=pred_dates, y=VaR_5, mode='lines', name='Pronóstico VaR 5% (GARCH)',
    line=dict(color='red')
))
# Pronóstico VaR 95%
fig.add_trace(go.Scatter(x=pred_dates, y=VaR_95, mode='lines', name='Pronóstico VaR 95% (GARCH)',
    line=dict(color='green')
))

Como venimos de un periodo de volatilidad reciente, el modelo GARCH "recuerda" esa agitación y expande las bandas de riesgo para los primeros días del pronóstico. Nuestro modelo detecta que cuando hay un pico grande, suelen seguirle otros picos grandes.

Interpretando estos resultados podemos ver que esperamos que en la próxima semana los retornos se mantengan mayoritariamente entre $-2.3%$ y $+2.1%$. Existe un riesgo del $5%$ de que suframos una pérdida diaria superior al $2.3%$, y dado que la memoria del mercado es alta, no esperamos que esta incertidumbre disminuya pronto.